# Notebook 18 - Pure ML and Cross-Attention Ablation Studies

Run the cells in order: Pure ML PCA/no-PCA, Feature Group, Feature 28, Modality Ablation, then Summary.

Pure ML uses the same radius-16 token windows as three-modality Cross-Attention. At every relative token position, ref and alt embeddings are concatenated; the 33 paired positions are flattened for tree classifiers, then bio/SpliceAI/geometry features are appended. PCA runs for all seeds before NoPCA. Train/validation feature matrices are prepared once as resumable disk-backed .npy memory maps; the OS can evict mapped pages under memory pressure, and the classifiers reuse the same files. Only one test split is loaded at a time. Runtime reports include matrix blocks, active token-window cache RAM, process RSS, available system RAM, and peak memory sampled during each tree fit. The full-feature Cross-Attention baseline is reused from Notebook 17.


## 1. Setup and configuration

Edit data and reference paths if needed. Set RESUME_EXISTING_BATCH to continue the Notebook 18 batch. Set BATCH_RUN_DIR_OVERRIDE to resume a specific output folder.


In [3]:
from datetime import datetime
from pathlib import Path
import importlib
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "core").is_dir() and (p / "controllers").is_dir()),
    _cwd,
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.module05v2_fusion_classifier.pipeline import FusionBatchPipeline
from core.module05v2_fusion_classifier.dataset import BIO_CORE_COLUMNS, SPLICEAI_COLUMNS
from core.module05v2_fusion_classifier import pipeline as pipeline_module
from core.module06_architecture_experiments.fm_ablation_preparation import (
    build_model_configs, build_split_configs, inspect_token_cache, extract_missing_token_caches,
)
from core.module06_architecture_experiments.fusion_classifier_benchmark import (
    load_legacy_best_classifier_params,
)
from core.module06_architecture_experiments.cross_attention_ablation_studies import (
    GEOM_FEATURES, feature_group_conditions, feature_28_conditions, modality_conditions,
    run_pure_ml_pca_study, run_cross_attention_ablation_study,
)

BASE_DIR = Path(r"F:\variant_data")
POOLED_CACHE_ROOT = BASE_DIR / "fm_embeddings" / "sequence_ablation"
TOKEN_CACHE_ROOT = BASE_DIR / "fm_embeddings" / "token_level_cache" / "module06"
GEOMETRY_K = 64
GEOMETRY_ROOT_NAME = "geometry_v2_k64_step2_matched"
GEOMETRY_ROOT = BASE_DIR / GEOMETRY_ROOT_NAME / "{context}"
FM_PROFILE_JSON = BASE_DIR / "profiling" / "fm_profiling_sequence_ablation.json"
DNA_MODEL_NAME = "nt_v1_500m_dna_1001"
PROTEIN_MODEL_NAME = "esm1b_650m_prot_501"
SEEDS = [5, 6, 7, 37, 42]
BATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PCA_COMPONENTS = 256
PURE_ML_MODES = {"PCA": True, "NoPCA": False}
PURE_ML_EXECUTION = {
    "prefer_gpu": True,
    "fallback_to_cpu": True,
    "cpu_threads": 16,
    "cpu_threads_pca": 16,
    "cpu_threads_no_pca": 4,
    "prefer_gpu_no_pca": False,
    "no_pca_max_bin": 128,  # CatBoost uses the equivalent border_count
    "gpu_device_id": 0,
    "gpu_ram_part": 0.80,
}

REFERENCE_BATCH_DIR = REPO_ROOT / "experiments" / "fusion_classifier_benchmark_20261120_192605"
REFERENCE_BENCHMARK_DIR = REFERENCE_BATCH_DIR / "module06_architecture_experiments" / "fusion_classifier_benchmark"
LEGACY_ML_BATCH_DIR = REPO_ROOT / "experiments" / "optimization_fixed_config_20261018_064727"

RESUME_EXISTING_BATCH = True
BATCH_RUN_DIR_OVERRIDE = r"D:\Missense_variant_pathogenicity_prediction\experiments\cross_attention_ablation_20261121_155305"
BATCH_POINTER = REPO_ROOT / "experiments" / "_cross_attention_ablation_latest_batch.json"
if BATCH_RUN_DIR_OVERRIDE is not None:
    BATCH_RUN_DIR = Path(BATCH_RUN_DIR_OVERRIDE).resolve()
elif RESUME_EXISTING_BATCH and BATCH_POINTER.is_file():
    BATCH_RUN_DIR = Path(json.loads(BATCH_POINTER.read_text(encoding="utf-8-sig"))["batch_run_dir"]).resolve()
else:
    BATCH_RUN_DIR = REPO_ROOT / "experiments" / f"cross_attention_ablation_{datetime.now():%Y%m%d_%H%M%S}"
    BATCH_POINTER.parent.mkdir(parents=True, exist_ok=True)
    BATCH_POINTER.write_text(
        json.dumps({"batch_run_dir": str(BATCH_RUN_DIR.resolve())}, indent=2),
        encoding="utf-8",
    )
BATCH_RUN_DIR.mkdir(parents=True, exist_ok=True)
MODULE06_OUTPUT_ROOT = BATCH_RUN_DIR / "module06_architecture_experiments"
MODULE06_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_CONFIG = {
    "attention_dropout": 0.18776012701077605,
    "dropout": 0.1955626033738947,
    "early_stop_patience": 6,
    "feature_dropout": 0.2507543060224074,
    "ffn_multiplier": 4,
    "k_radius": 16,
    "learning_rate": 0.00012206472463506827,
    "lr_factor": 0.5,
    "lr_patience": 3,
    "max_epochs": 30,
    "model_dim": 512,
    "num_heads": 8,
    "num_layers": 3,
    "pooling": "mean",
    "weight_decay": 3.380881049679085e-06,
    "batch_size": BATCH_SIZE,
}

_models = build_model_configs(dna_length=1001, protein_length=501)
DNA_MODEL = next(m for m in _models if m["name"] == DNA_MODEL_NAME)
PROTEIN_MODEL = next(m for m in _models if m["name"] == PROTEIN_MODEL_NAME)
SPLITS = build_split_configs(BASE_DIR)
DATASETS = [
    {"name": "Train3Val_Test", "train": "train3", "val": "val", "test": "test"},
    {"name": "Train3Val_ClinVarHQ", "train": "train3", "val": "val", "test": "clinvarhq"},
    {"name": "Train3Val_ProteinGym", "train": "train3", "val": "val", "test": "proteingym"},
    {"name": "Train3Val_UniProt", "train": "train3", "val": "val", "test": "uniprot"},
]
TARGET = {
    "name": "B_with_geometry",
    "ablation": "bio_core_dna_prot_spliceai_geom",
    "active_modalities": ["dna", "prot", "bio_core", "spliceai", "geom"],
    "bio_feature_names": [*BIO_CORE_COLUMNS, *SPLICEAI_COLUMNS],
    "geom_feature_names": list(GEOM_FEATURES),
    "geometry_dna_name": DNA_MODEL_NAME,
    "geometry_protein_name": PROTEIN_MODEL_NAME,
}

PIPELINE_CONFIG = {
    "batch_size": BATCH_SIZE,
    "epochs": MODEL_CONFIG["max_epochs"],
    "lr": MODEL_CONFIG["learning_rate"],
    "weight_decay": MODEL_CONFIG["weight_decay"],
    "lr_factor": MODEL_CONFIG["lr_factor"],
    "lr_patience": MODEL_CONFIG["lr_patience"],
    "early_stop_patience": MODEL_CONFIG["early_stop_patience"],
    "num_workers": 0,
    "seed": SEEDS[0],
    "xgb_n_jobs": 16,
}
pipeline = FusionBatchPipeline(
    base_dir=str(BASE_DIR),
    config=PIPELINE_CONFIG,
    datasets=DATASETS,
    models_space=[DNA_MODEL, PROTEIN_MODEL],
    pooling_strategies=["center"],
    experiments=[{"name": "CrossAttentionAblation", "type": "pytorch", "fusion": "cross_attention"}],
    explainability={"enable_shap": False, "enable_lime": False},
    fm_profile_json=str(FM_PROFILE_JSON),
    geom_profile_json=None,
    geometry_dir=str(GEOMETRY_ROOT),
    geom_feature_names=GEOM_FEATURES,
    batch_run_dir=str(BATCH_RUN_DIR),
    embedding_dir=str(POOLED_CACHE_ROOT),
)

print("Batch:", BATCH_RUN_DIR)
print("Reference Notebook 17 batch:", REFERENCE_BATCH_DIR)
print("Device:", DEVICE, "| seeds:", SEEDS)
print("DNA/Protein:", DNA_MODEL_NAME, "+", PROTEIN_MODEL_NAME)
print("Geometry: k=64 | features:", len(GEOM_FEATURES))
display(pd.DataFrame([MODEL_CONFIG]))

Batch: D:\Missense_variant_pathogenicity_prediction\experiments\cross_attention_ablation_20261121_155305
Reference Notebook 17 batch: D:\Missense_variant_pathogenicity_prediction\experiments\fusion_classifier_benchmark_20261120_192605
Device: cuda | seeds: [5, 6, 7, 37, 42]
DNA/Protein: nt_v1_500m_dna_1001 + esm1b_650m_prot_501
Geometry: k=64 | features: 8


,attention_dropout,dropout,early_stop_patience,feature_dropout,ffn_multiplier,k_radius,learning_rate,lr_factor,lr_patience,max_epochs,model_dim,num_heads,num_layers,pooling,weight_decay,batch_size
0,0.18776,0.195563,6,0.250754,4,16,0.000122,0.5,3,30,512,8,3,mean,0.000003,64


## 2. Preflight and baseline

Check required pooled-cache/geometry inputs for the pipeline, the radius-16 token caches used by Pure ML and Cross-Attention, and the full Cross-Attention baseline from Notebook 17. Stop if the baseline does not contain all five seeds and four test datasets.


In [4]:
from core.module05v2_fusion_classifier import pipeline as pipeline_module

os.environ["HF_HUB_OFFLINE"] = "1"
required_splits = [item["name"] for item in SPLITS]
pooled_audit = []
for split_name in required_splits:
    for model in (DNA_MODEL, PROTEIN_MODEL):
        path = POOLED_CACHE_ROOT / split_name / f"{model['name']}_center.pt"
        pooled_audit.append({
            "Split": split_name, "Model": model["name"],
            "Status": "complete" if path.is_file() else "missing", "Path": str(path),
        })
POOLED_AUDIT = pd.DataFrame(pooled_audit)
display(POOLED_AUDIT)
if (POOLED_AUDIT["Status"] != "complete").any():
    raise FileNotFoundError("Required center-pooled FM embeddings are missing.")

geometry_audit = []
geometry_root_train3 = str(GEOMETRY_ROOT).format(context="train3")
for split_name in required_splits:
    for model_name in (DNA_MODEL_NAME, PROTEIN_MODEL_NAME):
        path = pipeline_module._resolve_geom_path(geometry_root_train3, split_name, model_name, "center")
        geometry_audit.append({"Split": split_name, "Model": model_name, "Path": path})
display(pd.DataFrame(geometry_audit))

TOKEN_PREPARATION = extract_missing_token_caches(
    model_configs=[DNA_MODEL, PROTEIN_MODEL],
    split_configs=SPLITS,
    cache_root=TOKEN_CACHE_ROOT,
    batch_run_dir=BATCH_RUN_DIR,
    device=str(DEVICE),
)
TOKEN_CACHE_INDEX = TOKEN_PREPARATION["index"]
TOKEN_AUDIT = pd.DataFrame(inspect_token_cache(TOKEN_CACHE_ROOT, [DNA_MODEL, PROTEIN_MODEL], SPLITS))
display(TOKEN_AUDIT[["Model", "Split", "Status", "Shards", "Path"]])
if TOKEN_PREPARATION.get("status") != "complete" or (TOKEN_AUDIT["Status"] != "complete").any():
    raise RuntimeError("Token cache is incomplete; training did not start.")

baseline_path = REFERENCE_BENCHMARK_DIR / "seed_level_test_metrics.csv"
if not baseline_path.is_file():
    raise FileNotFoundError(f"Notebook 17 baseline test metrics not found: {baseline_path}")
baseline_all = pd.read_csv(baseline_path)
BASELINE_TEST = baseline_all[
    baseline_all["Target_Config"].astype(str).eq("B_with_geometry")
    & baseline_all["Fusion"].astype(str).eq("CrossAttention_ThreeModalities")
    & baseline_all["Classifier"].astype(str).eq("MLP")
].copy()
BASELINE_TEST["Seed"] = pd.to_numeric(BASELINE_TEST["Seed"], errors="coerce").astype("Int64")
expected_baseline = {(seed, ds["name"]) for seed in SEEDS for ds in DATASETS}
observed_baseline = set(zip(
    BASELINE_TEST["Seed"].dropna().astype(int),
    BASELINE_TEST["Dataset"].astype(str),
))
missing_baseline = sorted(expected_baseline - observed_baseline)
if missing_baseline:
    raise ValueError(f"Notebook 17 baseline is incomplete; missing seed/dataset rows: {missing_baseline}")
BASELINE_TEST = BASELINE_TEST[BASELINE_TEST["Seed"].isin(SEEDS)].copy()
BASELINE_TEST["MCC"] = pd.to_numeric(BASELINE_TEST["MCC"], errors="coerce")
BASELINE_TEST["AUROC"] = pd.to_numeric(BASELINE_TEST["AUROC"], errors="coerce")

LEGACY_BEST_PARAMS = load_legacy_best_classifier_params(
    LEGACY_ML_BATCH_DIR, target_config="B_with_geometry"
)
BENCHMARK_CONTEXT = {
    "pipeline": pipeline,
    "TARGET_CONFIGS": [TARGET],
    "datasets": DATASETS,
    "DNA_MODEL": DNA_MODEL,
    "PROTEIN_MODEL": PROTEIN_MODEL,
    "DEVICE": DEVICE,
    "token_embedding_cache_index": TOKEN_CACHE_INDEX,
    "token_window_radius": int(MODEL_CONFIG["k_radius"]),
    "token_batch_size": BATCH_SIZE,
    "allow_batch_size_resume": True,
    "baseline_source": str(baseline_path.resolve()),
    "input_data_signature": {
        "target_config": "B_with_geometry",
        "dna_model": DNA_MODEL_NAME,
        "protein_model": PROTEIN_MODEL_NAME,
        "geometry_k": GEOMETRY_K,
        "geometry_root": GEOMETRY_ROOT_NAME,
        "geometry_features": GEOM_FEATURES,
        "token_window_radius": int(MODEL_CONFIG["k_radius"]),
        "embedding_representation": "ref_alt_concat",
    },
}
print(f"Baseline verified: {len(BASELINE_TEST)} rows (expected {len(SEEDS) * len(DATASETS)}).")
display(BASELINE_TEST[["Seed", "Dataset", "MCC", "AUROC", "AUPRC", "Decision_Threshold"]].sort_values(["Dataset", "Seed"]))

,Split,Model,Status,Path
0,train3,nt_v1_500m_dna_1001,complete,F:\variant_data\fm_embeddings\sequence_ablatio...
1,train3,esm1b_650m_prot_501,complete,F:\variant_data\fm_embeddings\sequence_ablatio...
2,val,nt_v1_500m_dna_1001,complete,F:\variant_data\fm_embeddings\sequence_ablatio...
3,val,esm1b_650m_prot_501,complete,F:\variant_data\fm_embeddings\sequence_ablatio...
4,test,nt_v1_500m_dna_1001,complete,F:\variant_data\fm_embeddings\sequence_ablatio...
5,test,esm1b_650m_prot_501,complete,F:\variant_data\fm_embeddings\sequence_ablatio...
6,clinvarhq,nt_v1_500m_dna_1001,complete,F:\variant_data\fm_embeddings\sequence_ablatio...
7,clinvarhq,esm1b_650m_prot_501,complete,F:\variant_data\fm_embeddings\sequence_ablatio...
8,proteingym,nt_v1_500m_dna_1001,complete,F:\variant_data\fm_embeddings\sequence_ablatio...
9,proteingym,esm1b_650m_prot_501,complete,F:\variant_data\fm_embeddings\sequence_ablatio...


,Split,Model,Path
0,train3,nt_v1_500m_dna_1001,F:\variant_data\geometry_v2_k64_step2_matched\...
1,train3,esm1b_650m_prot_501,F:\variant_data\geometry_v2_k64_step2_matched\...
2,val,nt_v1_500m_dna_1001,F:\variant_data\geometry_v2_k64_step2_matched\...
3,val,esm1b_650m_prot_501,F:\variant_data\geometry_v2_k64_step2_matched\...
4,test,nt_v1_500m_dna_1001,F:\variant_data\geometry_v2_k64_step2_matched\...
5,test,esm1b_650m_prot_501,F:\variant_data\geometry_v2_k64_step2_matched\...
6,clinvarhq,nt_v1_500m_dna_1001,F:\variant_data\geometry_v2_k64_step2_matched\...
7,clinvarhq,esm1b_650m_prot_501,F:\variant_data\geometry_v2_k64_step2_matched\...
8,proteingym,nt_v1_500m_dna_1001,F:\variant_data\geometry_v2_k64_step2_matched\...
9,proteingym,esm1b_650m_prot_501,F:\variant_data\geometry_v2_k64_step2_matched\...


,Model,Split,Status,Shards,Path
0,nt_v1_500m_dna_1001,train3,complete,11604,F:\variant_data\fm_embeddings\token_level_cach...
1,nt_v1_500m_dna_1001,val,complete,6851,F:\variant_data\fm_embeddings\token_level_cach...
2,nt_v1_500m_dna_1001,test,complete,3048,F:\variant_data\fm_embeddings\token_level_cach...
3,nt_v1_500m_dna_1001,clinvarhq,complete,352,F:\variant_data\fm_embeddings\token_level_cach...
4,nt_v1_500m_dna_1001,proteingym,complete,736,F:\variant_data\fm_embeddings\token_level_cach...
5,nt_v1_500m_dna_1001,uniprot,complete,667,F:\variant_data\fm_embeddings\token_level_cach...
6,esm1b_650m_prot_501,train3,complete,1451,F:\variant_data\fm_embeddings\token_level_cach...
7,esm1b_650m_prot_501,val,complete,857,F:\variant_data\fm_embeddings\token_level_cach...
8,esm1b_650m_prot_501,test,complete,381,F:\variant_data\fm_embeddings\token_level_cach...
9,esm1b_650m_prot_501,clinvarhq,complete,44,F:\variant_data\fm_embeddings\token_level_cach...


Baseline verified: 20 rows (expected 20).


,Seed,Dataset,MCC,AUROC,AUPRC,Decision_Threshold
401,5,Train3Val_ClinVarHQ,0.8788,0.9899,0.9901,0.888212
417,6,Train3Val_ClinVarHQ,0.8664,0.9884,0.9894,0.918221
433,7,Train3Val_ClinVarHQ,0.8792,0.9894,0.9893,0.559536
449,37,Train3Val_ClinVarHQ,0.8943,0.9894,0.9887,0.479132
465,42,Train3Val_ClinVarHQ,0.8867,0.9896,0.9902,0.814246
402,5,Train3Val_ProteinGym,0.6963,0.9265,0.9191,0.888212
418,6,Train3Val_ProteinGym,0.6718,0.9072,0.8802,0.918221
434,7,Train3Val_ProteinGym,0.6942,0.9307,0.9216,0.559536
450,37,Train3Val_ProteinGym,0.6745,0.9236,0.9133,0.479132
466,42,Train3Val_ProteinGym,0.6702,0.9182,0.9064,0.814246


## 3. Pure ML - PCA and no PCA

All seeds run the three PCA classifiers first; completed PCA seed/mode results are skipped on resume. Then all seeds run NoPCA. Train and validation matrices are prepared once as resumable disk-backed .npy memory maps and reused across classifiers, modes, and seeds; this replaces the 23+ GiB anonymous feature arrays with mappings whose pages the OS can reclaim under memory pressure. Test matrices are loaded one split at a time after fitting, shared across pending classifiers, then released. Runtime reports show feature blocks, active token-window RAM, process RSS and available RAM; model fits are sampled for peak host and GPU memory. The sequence input is not center-pooled: it uses 33 token positions around the variant, concatenates ref/alt at each position, then flattens positions in order. Bio/SpliceAI/geometry features are appended. PCA is fit separately to the DNA-window and protein-window blocks; the tabular features are preserved.


In [ ]:
PURE_ML_RUN_PLAN = pd.DataFrame([
    {"Input_Representation": "window_radius16_ref_alt_concat_flatten", "PCA_Mode": mode, "Use_PCA": use_pca, "Classifier": classifier, "Seed": seed}
    for mode, use_pca in PURE_ML_MODES.items()
    for classifier in ("XGBoost", "CatBoost", "LightGBM")
    for seed in SEEDS
])
print(f"Pure ML plan: {len(PURE_ML_RUN_PLAN)} fits on radius16 local token ref_alt_concat = 2 PCA modes x 3 classifiers x 5 seeds")
display(PURE_ML_RUN_PLAN)

PURE_ML_RESULTS = run_pure_ml_pca_study(
    context=BENCHMARK_CONTEXT,
    target=TARGET,
    seeds=SEEDS,
    classifier_parameters=LEGACY_BEST_PARAMS,
    output_root=MODULE06_OUTPUT_ROOT,
    pca_components=PCA_COMPONENTS,
    pca_modes=PURE_ML_MODES,
    execution_config=PURE_ML_EXECUTION,
    display_fn=print,
)
display(PURE_ML_RESULTS.sort_values(["Dataset", "Classifier", "PCA_Mode", "Seed"]))

Pure ML plan: 30 fits on radius16 local token ref_alt_concat = 2 PCA modes x 3 classifiers x 5 seeds


,Input_Representation,PCA_Mode,Use_PCA,Classifier,Seed
0,window_radius16_ref_alt_concat_flatten,PCA,True,XGBoost,5
1,window_radius16_ref_alt_concat_flatten,PCA,True,XGBoost,6
2,window_radius16_ref_alt_concat_flatten,PCA,True,XGBoost,7
3,window_radius16_ref_alt_concat_flatten,PCA,True,XGBoost,37
4,window_radius16_ref_alt_concat_flatten,PCA,True,XGBoost,42
5,window_radius16_ref_alt_concat_flatten,PCA,True,CatBoost,5
6,window_radius16_ref_alt_concat_flatten,PCA,True,CatBoost,6
7,window_radius16_ref_alt_concat_flatten,PCA,True,CatBoost,7
8,window_radius16_ref_alt_concat_flatten,PCA,True,CatBoost,37
9,window_radius16_ref_alt_concat_flatten,PCA,True,CatBoost,42


[Resources] before Pure ML feature preload | process_RSS=0.79 GiB | process_peak=0.79 GiB | system_available=83.16 GiB | CUDA_free=14.63/15.89 GiB
[Pure ML] Preloading radius-16 train/validation feature matrices into disk-backed mmap files...
[Pure ML mmap] Creating disk-backed feature matrix for train3...
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio_core', 'spliceai', 'geom']
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.



Window cache dna/train3: 100%|██████████| 11604/11604 [00:06<00:00, 1825.83shard/s]


[Window cache] dna/train3: 23208 samples ready in 6.5s


Window cache protein/train3: 100%|██████████| 1451/1451 [00:01<00:00, 873.61shard/s]


... (101 outputs left out)

Window cache dna/uniprot: 100%|██████████| 667/667 [00:00<00:00, 1789.23shard/s]


[Window cache] dna/uniprot: 1333 samples ready in 0.4s


Window cache protein/uniprot: 100%|██████████| 84/84 [00:00<00:00, 955.36shard/s]


[Window cache] protein/uniprot: 1333 samples ready in 0.1s
[Pure ML RAM] uniprot: samples=1,333, features=169,054; matrix=0.84 GiB (DNA=0.42, protein=0.42, bio=0.000, geometry=0.000); active token-window tensors=0.42 GiB
[Resources] after assembling uniprot | process_RSS=27.95 GiB | process_peak=42.02 GiB | system_available=55.89 GiB | CUDA_free=14.56/15.89 GiB
[Pure ML] XGBoost/PCA/seed_42/Train3Val_UniProt: MCC=0.4809
[Pure ML] CatBoost/PCA/seed_42/Train3Val_UniProt: MCC=0.4617
[Pure ML] LightGBM/PCA/seed_42/Train3Val_UniProt: MCC=0.4945
[Pure ML] Reusing preloaded feature matrices from RAM for seed 5.
[Resources] before XGBoost/NoPCA/seed_5 fit | process_RSS=27.07 GiB | process_peak=42.02 GiB | system_available=56.88 GiB | CUDA_free=14.56/15.89 GiB


## 4. Feature Group Ablation - Cross-Attention 3 modalities

Remove one feature group at a time. Train each ablation for each seed. Reuse the All 28 baseline from Notebook 17.


In [ ]:
FEATURE_GROUP_RESULTS = run_cross_attention_ablation_study(
    context=BENCHMARK_CONTEXT,
    base_target=TARGET,
    conditions=feature_group_conditions(TARGET),
    seeds=SEEDS,
    model_config=MODEL_CONFIG,
    output_root=MODULE06_OUTPUT_ROOT,
    study="feature_group_ablation",
    baseline_test=BASELINE_TEST,
    display_fn=print,
)
display(FEATURE_GROUP_RESULTS["summary"])

## 5. Feature 28 Ablation - Cross-Attention 3 modalities

Remove one of the 28 tabular or geometry features at a time. Reuse the None reference from Notebook 17 and calculate deltas by matching seed and test dataset.


In [ ]:
FEATURE_28_RESULTS = run_cross_attention_ablation_study(
    context=BENCHMARK_CONTEXT,
    base_target=TARGET,
    conditions=feature_28_conditions(TARGET),
    seeds=SEEDS,
    model_config=MODEL_CONFIG,
    output_root=MODULE06_OUTPUT_ROOT,
    study="feature_28_ablation",
    baseline_test=BASELINE_TEST,
    display_fn=print,
)
display(FEATURE_28_RESULTS["summary"])

## 6. Modality Ablation - active Cross-Attention branches

DNA, Protein and Bio/Geometry are the three branches. Keep cross-attention between the branches that remain. If only one branch remains, send its representation directly to the same classifier MLP. Reuse All Modalities from Notebook 17.


In [ ]:
MODALITY_RESULTS = run_cross_attention_ablation_study(
    context=BENCHMARK_CONTEXT,
    base_target=TARGET,
    conditions=modality_conditions(TARGET),
    seeds=SEEDS,
    model_config=MODEL_CONFIG,
    output_root=MODULE06_OUTPUT_ROOT,
    study="modality_ablation",
    baseline_test=BASELINE_TEST,
    display_fn=print,
)
display(MODALITY_RESULTS["summary"])

## 7. Summary tables and plots

Save per-seed test rows, mean/std by condition and dataset, deltas against baseline, and MCC plots. Keep the Pure ML classifiers separate in the summaries.


In [ ]:
from pathlib import Path

summary_root = MODULE06_OUTPUT_ROOT / "summaries"
summary_root.mkdir(parents=True, exist_ok=True)

pure_path = MODULE06_OUTPUT_ROOT / "pure_ml_ref_alt_concat_window33_gpu_first_pca16_nopca8" / "test_metrics_by_seed.csv"
pure = pd.read_csv(pure_path) if pure_path.is_file() else pd.DataFrame()
group_path = MODULE06_OUTPUT_ROOT / "feature_group_ablation" / "delta_vs_baseline_by_seed.csv"
feature_path = MODULE06_OUTPUT_ROOT / "feature_28_ablation" / "delta_vs_baseline_by_seed.csv"
modality_path = MODULE06_OUTPUT_ROOT / "modality_ablation" / "delta_vs_baseline_by_seed.csv"
group_delta = pd.read_csv(group_path) if group_path.is_file() else pd.DataFrame()
feature_delta = pd.read_csv(feature_path) if feature_path.is_file() else pd.DataFrame()
modality_delta = pd.read_csv(modality_path) if modality_path.is_file() else pd.DataFrame()

if not pure.empty:
    pure_summary = pure.groupby(["Classifier", "PCA_Mode", "Dataset"], dropna=False)[
        ["Accuracy", "F1_Score", "MCC", "AUROC", "AUPRC"]
    ].agg(["mean", "std", "count"])
    pure_summary.columns = [f"{metric}_{stat}" for metric, stat in pure_summary.columns]
    pure_summary = pure_summary.reset_index()
    pure_summary.to_csv(summary_root / "pure_ml_ref_alt_concat_window33_gpu_first_pca16_nopca8_mean_std_by_classifier_pca_dataset.csv", index=False)
    display(pure_summary.sort_values(["Dataset", "MCC_mean"], ascending=[True, False]))

all_delta = pd.concat(
    [frame for frame in (group_delta, feature_delta, modality_delta) if not frame.empty],
    ignore_index=True,
) if any(not frame.empty for frame in (group_delta, feature_delta, modality_delta)) else pd.DataFrame()
if not all_delta.empty:
    all_delta.to_csv(summary_root / "cross_attention_ablation_delta_by_seed.csv", index=False)
    delta_summary = all_delta.groupby(["Study", "Condition", "Dataset"], dropna=False)[
        ["MCC", "AUROC", "Baseline_MCC", "Baseline_AUROC", "Delta_MCC", "Delta_AUROC"]
    ].agg(["mean", "std", "count"])
    delta_summary.columns = [f"{metric}_{stat}" for metric, stat in delta_summary.columns]
    delta_summary = delta_summary.reset_index()
    delta_summary.to_csv(summary_root / "cross_attention_ablation_delta_mean_std.csv", index=False)
    display(delta_summary.sort_values(["Study", "Dataset", "Delta_MCC_mean"], ascending=[True, True, False]))

if not all_delta.empty:
    plot_data = all_delta.groupby(["Study", "Condition", "Dataset"], dropna=False)["Delta_MCC"].mean().reset_index()
    for study_name, frame in plot_data.groupby("Study"):
        pivot = frame.pivot(index="Condition", columns="Dataset", values="Delta_MCC")
        ax = pivot.plot(kind="bar", figsize=(15, max(5, 0.35 * len(pivot))))
        ax.axhline(0, color="black", linewidth=1)
        ax.set_title(f"{study_name}: mean Delta MCC vs full Cross-Attention baseline")
        ax.set_ylabel("Delta MCC")
        ax.set_xlabel("")
        plt.xticks(rotation=60, ha="right")
        plt.tight_layout()
        plt.show()


# One per-run global table combines all available rows without averaging models together.
cross_attention_parts = []
for study_name in ("feature_group_ablation", "feature_28_ablation", "modality_ablation"):
    metric_path = MODULE06_OUTPUT_ROOT / study_name / "test_metrics_by_seed.csv"
    if metric_path.is_file():
        frame = pd.read_csv(metric_path)
        if not frame.empty:
            cross_attention_parts.append(frame)
baseline_global = BASELINE_TEST.copy()
baseline_global["Study"] = "baseline"
baseline_global["Condition"] = "All_28_All_Modalities"
baseline_global["Architecture"] = "CrossAttention_ThreeModalities"
baseline_global["Branch"] = "DL"
pure_global = pure.copy()
if not pure_global.empty:
    pure_global["Branch"] = "Pure_ML"
    pure_global["Study"] = "pure_ml_ref_alt_concat_window33_gpu_first_pca16_nopca8"
    pure_global["Condition"] = pure_global["PCA_Mode"]
    pure_global["Architecture"] = pure_global["Classifier"]
global_frames = [baseline_global, *cross_attention_parts]
if not pure_global.empty:
    global_frames.append(pure_global)
global_metrics = pd.concat(global_frames, ignore_index=True, sort=False)
global_metrics.to_csv(summary_root / "global_test_metrics_by_seed.csv", index=False)
display(global_metrics.sort_values(["Dataset", "Study", "Condition", "Seed"]))

if not pure.empty:
    pure_plot = pure_summary.copy()
    pure_plot["Model"] = pure_plot["Classifier"].astype(str) + "_" + pure_plot["PCA_Mode"].astype(str)
    for dataset_name, frame in pure_plot.groupby("Dataset"):
        ax = frame.set_index("Model")["MCC_mean"].sort_values(ascending=False).plot(
            kind="bar", figsize=(10, 4), title=f"Pure ML mean MCC ? {dataset_name}"
        )
        ax.set_ylabel("MCC mean across five seeds")
        ax.set_xlabel("")
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()
        plt.show()

print("Global test metrics:", summary_root / "global_test_metrics_by_seed.csv")
print("Summary output:", summary_root)


## 8. Cross-Attention 3 modalities - model summary

Build the model with the active B configuration and print its structure, parameter counts, and input/output shapes. This cell initializes the model and performs one inference-only dummy forward pass; it does not train, evaluate test data, or write experiment results.


In [5]:
import torch
from core.module06_architecture_experiments.cross_attention_3modal_experiment import (
    CrossAttentionThreeModalities,
)

# Hidden dimensions are read from the token-cache index produced in the preflight cell.
_token_models = TOKEN_CACHE_INDEX["models"]
_summary_dimensions = {
    "token_dna": int(_token_models[DNA_MODEL_NAME]["train3"]["hidden_size"]),
    "token_protein": int(_token_models[PROTEIN_MODEL_NAME]["train3"]["hidden_size"]),
    "bio": len(TARGET["bio_feature_names"]),
    "geometry": len(TARGET["geom_feature_names"]),
}
_summary_model = CrossAttentionThreeModalities(
    dna_dim=_summary_dimensions["token_dna"],
    protein_dim=_summary_dimensions["token_protein"],
    bio_dim=_summary_dimensions["bio"],
    geom_dim=_summary_dimensions["geometry"],
    config=MODEL_CONFIG,
).to(DEVICE).eval()

print("Cross-Attention 3 modalities | Notebook 18 | B_with_geometry")
print(f"DNA FM: {DNA_MODEL_NAME} (token hidden={_summary_dimensions['token_dna']})")
print(f"Protein FM: {PROTEIN_MODEL_NAME} (token hidden={_summary_dimensions['token_protein']})")
print(
    "Inputs: DNA REF/ALT tokens + Protein REF/ALT tokens, "
    f"radius={MODEL_CONFIG['k_radius']} (window={2 * MODEL_CONFIG['k_radius'] + 1}), "
    f"{_summary_dimensions['bio']} bio/SpliceAI features + "
    f"{_summary_dimensions['geometry']} geometry features"
)
print("\nConfiguration:")
display(pd.DataFrame([MODEL_CONFIG]))

print("\nModule structure:")
print(_summary_model)

print("\nParameter counts:")
_summary_rows = []
for _name, _module in _summary_model.named_children():
    _summary_rows.append({
        "Module": _name,
        "Parameters": sum(p.numel() for p in _module.parameters()),
        "Trainable": sum(p.numel() for p in _module.parameters() if p.requires_grad),
    })
_summary_rows.append({
    "Module": "TOTAL",
    "Parameters": sum(p.numel() for p in _summary_model.parameters()),
    "Trainable": sum(p.numel() for p in _summary_model.parameters() if p.requires_grad),
})
display(pd.DataFrame(_summary_rows))

# One inference-only dummy pass confirms the actual batch input and output shapes.
_summary_batch = 1
_summary_width = 2 * int(MODEL_CONFIG["k_radius"]) + 1
_summary_dna = torch.zeros(
    (_summary_batch, 2, _summary_width, _summary_dimensions["token_dna"]),
    device=DEVICE,
)
_summary_protein = torch.zeros(
    (_summary_batch, 2, _summary_width, _summary_dimensions["token_protein"]),
    device=DEVICE,
)
_summary_mask_dna = torch.ones(
    (_summary_batch, 2, _summary_width), dtype=torch.bool, device=DEVICE
)
_summary_mask_protein = torch.ones_like(_summary_mask_dna)
_summary_variant_index = torch.full(
    (_summary_batch, 2), int(MODEL_CONFIG["k_radius"]), dtype=torch.long, device=DEVICE
)
_summary_bio = torch.zeros((_summary_batch, _summary_dimensions["bio"]), device=DEVICE)
_summary_geometry = torch.zeros(
    (_summary_batch, _summary_dimensions["geometry"]), device=DEVICE
)
with torch.inference_mode():
    _summary_logits = _summary_model(
        None, None, _summary_bio, _summary_geometry,
        _summary_dna, _summary_mask_dna, _summary_variant_index,
        _summary_protein, _summary_mask_protein, _summary_variant_index,
    )
print("\nDummy forward shapes:")
print(f"DNA tokens:      {tuple(_summary_dna.shape)}")
print(f"Protein tokens:  {tuple(_summary_protein.shape)}")
print(f"Bio features:    {tuple(_summary_bio.shape)}")
print(f"Geometry:        {tuple(_summary_geometry.shape)}")
print(f"Output logits:   {tuple(_summary_logits.shape)}")


Cross-Attention 3 modalities | Notebook 18 | B_with_geometry
DNA FM: nt_v1_500m_dna_1001 (token hidden=1280)
Protein FM: esm1b_650m_prot_501 (token hidden=1280)
Inputs: DNA REF/ALT tokens + Protein REF/ALT tokens, radius=16 (window=33), 20 bio/SpliceAI features + 8 geometry features

Configuration:


,attention_dropout,dropout,early_stop_patience,feature_dropout,ffn_multiplier,k_radius,learning_rate,lr_factor,lr_patience,max_epochs,model_dim,num_heads,num_layers,pooling,weight_decay,batch_size
0,0.18776,0.195563,6,0.250754,4,16,0.000122,0.5,3,30,512,8,3,mean,0.000003,64



Module structure:
CrossAttentionThreeModalities(
  (dna_window): _VariantCenteredWindow(
    (projection): Linear(in_features=2560, out_features=512, bias=True)
    (relative_position_embedding): Embedding(129, 512)
    (dropout): Dropout(p=0.1955626033738947, inplace=False)
  )
  (protein_window): _VariantCenteredWindow(
    (projection): Linear(in_features=2560, out_features=512, bias=True)
    (relative_position_embedding): Embedding(129, 512)
    (dropout): Dropout(p=0.1955626033738947, inplace=False)
  )
  (bio_tokenizer): _ScalarFeatureTokenizer(
    (dropout): Dropout(p=0.2507543060224074, inplace=False)
  )
  (geom_tokenizer): _ScalarFeatureTokenizer(
    (dropout): Dropout(p=0.2507543060224074, inplace=False)
  )
  (cross_layers): ModuleList(
    (0-2): 3 x ModuleDict(
      (dna): _CrossAttentionBlock(
        (query_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (context_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (cross_attent

,Module,Parameters,Trainable
0,dna_window,1377792,1377792
1,protein_window,1377792,1377792
2,bio_tokenizer,20480,20480
3,geom_tokenizer,8192,8192
4,cross_layers,28380672,28380672
5,pool_scores,4611,4611
6,output_norm,3072,3072
7,classifier,2099201,2099201
8,TOTAL,33273860,33273860



Dummy forward shapes:
DNA tokens:      (1, 2, 33, 1280)
Protein tokens:  (1, 2, 33, 1280)
Bio features:    (1, 20)
Geometry:        (1, 8)
Output logits:   (1,)
